In [ ]:
# Get data from: https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page 
from framechain.operations.io import LocalFileReader
from framechain.operations.pandas.record_extractors.delimited import DelimitedRecordExtractor, NumberField, TimestampField, IntegerField, StringField

## Extracting typed columns from the raw CSV

`DelimitedRecordExtractor` declares a typed schema for the CSV: each `Field` (`IntegerField`, `TimestampField`, `NumberField`, ...) parses and casts one source column and maps it onto an output column name. Composing `LocalFileReader() >> record_extractor` produces a single pipeline stage that reads the file and returns a fully typed DataFrame in one call, instead of a raw `pandas.read_csv()` load followed by ad-hoc type coercion.


In [ ]:
record_extractor = DelimitedRecordExtractor(
    fields=[
        IntegerField('vendor_id', column_id='VendorID', size=8),
        TimestampField('pickup_timestamp', column_id='tpep_pickup_datetime'),
        TimestampField('dropoff_timestamp', column_id='tpep_dropoff_datetime'),
        IntegerField('passenger_count', size=8),
        NumberField('distance', column_id='trip_distance'),
        IntegerField('pickup_location_id', column_id='PULocationID'),
        IntegerField('dropoff_location_id', column_id='DOLocationID'),
        IntegerField('payment_type', size=8),
        NumberField('fare_amount'),
        NumberField('tip_amount'),
        NumberField('congestion_surcharge'),
    ],
)
read_extract_records = LocalFileReader() >> record_extractor

## Getting the data

Download a month of the NYC Yellow Taxi trip data CSV from the link in the first cell (any month works) and place it at `data/yellow_tripdata_2021-01.csv` relative to this notebook. If you download a different month, adjust the filename in the cell below to match.


In [ ]:
input_file = 'data/yellow_tripdata_2021-01.csv'
df = read_extract_records(input_file)
df

## Inspecting the extracted data

Because extraction is driven by the typed field schema, the resulting DataFrame already has the dtypes declared by each `Field` (timestamps, nullable integers, floats) rather than the objects/strings `pandas.read_csv` would infer unassisted.


In [ ]:
df.dtypes

## Filtering bad rows and computing derived columns

`DropRows` discards records with a null `vendor_id` or zero `passenger_count`. `trip_duration` is derived from the pickup/dropoff timestamps, and a 10% surcharge is applied to `fare_amount` for card payments (`payment_type == 2`).

The surcharge uses `DFWhere` rather than a plain boolean-mask assignment (`df.loc[mask, 'fare_amount'] *= 1.1`) because `DFWhere` is itself a composable `Operation`: it infers the columns required and affected by the wrapped `SetColumn`, slices the DataFrame down to just those columns for the masked rows, runs the transform, and integrates the result back — so it only ever touches the data it needs to, and participates in the same profiling/graph tooling as the rest of the pipeline.


In [ ]:
from framechain.operations.pandas import DropRows, Column, IsNull, TimedeltaToSeconds, SetColumn, DFWhere
remove_bad_data = DropRows((Column('vendor_id') >> IsNull()) | (Column('passenger_count') == 0))

calculate_duration = SetColumn('trip_duration', (Column('dropoff_timestamp') - Column('pickup_timestamp')) >> TimedeltaToSeconds())

add_card_surcharge = DFWhere(Column('payment_type') == 2, SetColumn('fare_amount', Column('fare_amount')*1.1))

In [ ]:
calculate_cost_per_passenger_per_minute = SetColumn('cost_per_passenger_per_minute', (Column('fare_amount') + Column('tip_amount'))/(Column('passenger_count') * Column('trip_duration')/60))

## Chaining and running the transforms

The individual operations compose with `>>` into a single `transforms` operation, which can be applied directly to the DataFrame produced above to validate the whole transform stage before wiring it into the full pipeline.


In [ ]:
transforms = remove_bad_data >> calculate_duration >> add_card_surcharge >> calculate_cost_per_passenger_per_minute

In [ ]:
transforms(df)

## Running and profiling the full pipeline

`full_pipeline` chains extraction and transforms into a single operation that runs directly against the input CSV, end-to-end. `profile_snakeviz()` executes it under `cProfile` and opens the result in SnakeViz, giving a flame-graph-style breakdown of time spent in each operation in the chain — useful for spotting the slow stage without instrumenting the code by hand.


In [ ]:
full_pipeline = read_extract_records >> transforms
full_pipeline.profile_snakeviz(input_file)